In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import sys
import os

In [2]:
sys.path.append(os.path.join(os.getcwd(), "..", "src"))

from data_loader import DataLoader
from feature_engineering import FeatureEngineering
from preprocessing import Preprocessing
from train import Trainer
from evaluate import Evaluator
from predict import Predictor

from sklearn.model_selection import train_test_split

### **Loading raw data**

In [3]:
loader = DataLoader(subfolder="raw")
loader.load("ev_market_2026.csv").validate().info()
df = loader.get_df()

2026-06-13 14:29:19 | INFO     | data_loader | DataLoader initialized | subfolder: raw
2026-06-13 14:29:19 | INFO     | data_loader | Loading file | path: c:\Users\mrcoo\OneDrive\Desktop\ev-price-prediction\data\raw\ev_market_2026.csv
2026-06-13 14:29:19 | INFO     | data_loader | file loaded successfully | shape: (2000, 24) | columns: ['brand', 'model', 'year', 'variant', 'price_usd', 'battery_capacity_kwh', 'range_miles', 'charging_speed_kw', 'acceleration_0_60_mph', 'top_speed_mph', 'horsepower', 'torque_nm', 'drive_type', 'seating_capacity', 'body_type', 'cargo_volume_cubic_ft', 'weight_kg', 'safety_rating', 'autopilot_level', 'country_of_origin', 'market_segment', 'annual_sales_units', 'customer_rating', 'warranty_years'] 
2026-06-13 14:29:19 | INFO     | data_loader | Running validation checks...
2026-06-13 14:29:19 | WARNING  | data_loader | Columns with missing values:
Series([], )
2026-06-13 14:29:19 | INFO     | data_loader | No duplicate rows found
2026-06-13 14:29:19 | INFO

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   brand                  2000 non-null   str    
 1   model                  2000 non-null   str    
 2   year                   2000 non-null   int64  
 3   variant                2000 non-null   str    
 4   price_usd              2000 non-null   float64
 5   battery_capacity_kwh   2000 non-null   float64
 6   range_miles            2000 non-null   float64
 7   charging_speed_kw      2000 non-null   float64
 8   acceleration_0_60_mph  2000 non-null   float64
 9   top_speed_mph          2000 non-null   float64
 10  horsepower             2000 non-null   float64
 11  torque_nm              2000 non-null   float64
 12  drive_type             2000 non-null   str    
 13  seating_capacity       2000 non-null   int64  
 14  body_type              2000 non-null   str    
 15  cargo_volume_cu

### **2. Split into train / test  (before ANY processing — no leakage)**

In [5]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print(f"\nSplit → train: {train_df.shape} | test: {test_df.shape}")


Split → train: (1600, 24) | test: (400, 24)


### **3. Feature engineering**

In [6]:
fe_train = FeatureEngineering(train_df)
train_df = (fe_train
            .add_efficiency_features()
            .add_performance_score()
            .add_age_feature()
            .add_value_score()
            .get_df())

fe_test = FeatureEngineering(test_df)
test_df = (fe_test
           .add_efficiency_features()
           .add_performance_score()
           .add_age_feature()
           .add_value_score()
           .get_df())

print(f"After feature engineering → train: {train_df.shape} | test: {test_df.shape}")


2026-06-13 14:29:19 | INFO     | feature_engineering | FeatureEngineering initialized | shape: (1600, 24)
2026-06-13 14:29:19 | INFO     | feature_engineering | Adding efficiency features...
2026-06-13 14:29:19 | INFO     | feature_engineering | Adding performance score...
2026-06-13 14:29:19 | INFO     | feature_engineering | Adding vehicle age feature...
2026-06-13 14:29:19 | INFO     | feature_engineering | Adding value score feature...
2026-06-13 14:29:19 | INFO     | feature_engineering | Returning feature-engineered dataframe | shape: (1600, 29)
2026-06-13 14:29:19 | INFO     | feature_engineering | New features added: ['range_per_kwh', 'charge_rate', 'performance_score', 'vehicle_age', 'value_score']
2026-06-13 14:29:19 | INFO     | feature_engineering | FeatureEngineering initialized | shape: (400, 24)
2026-06-13 14:29:19 | INFO     | feature_engineering | Adding efficiency features...
2026-06-13 14:29:19 | INFO     | feature_engineering | Adding performance score...
2026-06-13

### **4. Preprocessing  (fit on train, transform test)**

In [7]:
# ── 4. Preprocessing ──────────────────────────────────────────────────────────
# Fit ONLY on train: Transform on test:
# Dropped high-cardinality ID-like columns that don't generalise.

DROP_COLS = ["brand", "model"]    # too many unique values to encode meaningfully
train_df = train_df.drop(columns=DROP_COLS, errors="ignore")
test_df  = test_df.drop(columns=DROP_COLS,  errors="ignore")

preprocessor = Preprocessing(train_df)
preprocessor.handling_missing_values() \
             .encoding(target_col="price_usd") \
             .scaling(target_col="price_usd")

train_clean = preprocessor.get_df()
test_clean  = preprocessor.transform(test_df)

print(f"After preprocessing → train: {train_clean.shape} | test: {test_clean.shape}")



2026-06-13 14:29:19 | INFO     | preprocessing | Preprocessing initialized | shape (1600, 27)
2026-06-13 14:29:19 | INFO     | preprocessing | Handling missing values...
2026-06-13 14:29:19 | INFO     | preprocessing | Missing values handled successfully
2026-06-13 14:29:19 | INFO     | preprocessing | Encoding categorical columns...
2026-06-13 14:29:19 | INFO     | preprocessing | Encoding complete
2026-06-13 14:29:19 | INFO     | preprocessing | Scaling numerical columns...
2026-06-13 14:29:19 | INFO     | preprocessing | Scaling complete | 35 columns scaled
2026-06-13 14:29:19 | INFO     | preprocessing | Returning preprocessed dataframe | shape: (1600, 36)
2026-06-13 14:29:19 | INFO     | preprocessing | Transforming new data | input shape: (400, 27)
2026-06-13 14:29:19 | INFO     | preprocessing | Scaling applied | 35 columns scaled
2026-06-13 14:29:19 | INFO     | preprocessing | Transform complete | output shape: (400, 36)
After preprocessing → train: (1600, 36) | test: (400, 36

### **5. Train models + compare**

In [8]:
trainer = Trainer(train_clean, test_clean, target_col="price_usd")
trainer.train_all()

print("\n── Model Comparison ─────────────────────────────")
print(trainer.get_results().to_string())
print("─────────────────────────────────────────────────\n")

2026-06-13 14:29:19 | INFO     | train | Trainer initialized | train: (1600, 35) | test: (400, 35) | target: price_usd
2026-06-13 14:29:19 | INFO     | train | Training 3 models...
2026-06-13 14:29:19 | INFO     | train | Training 'ridge'...
2026-06-13 14:29:19 | INFO     | train | [ridge] MAE=7,401.86 | RMSE=11,155.53 | R²=0.8909
2026-06-13 14:29:19 | INFO     | train | Training 'random_forest'...
2026-06-13 14:29:20 | INFO     | train | [random_forest] MAE=3,593.20 | RMSE=5,144.69 | R²=0.9768
2026-06-13 14:29:20 | INFO     | train | Training 'gradient_boosting'...
2026-06-13 14:29:21 | INFO     | train | [gradient_boosting] MAE=2,742.67 | RMSE=3,937.38 | R²=0.9864
2026-06-13 14:29:21 | INFO     | train | Best model: gradient_boosting | R²=0.9864 | RMSE=3937.3821

── Model Comparison ─────────────────────────────
                         MAE        RMSE      R2
model                                           
gradient_boosting  2742.6668   3937.3821  0.9864
random_forest      3593.201

### **7. Evaluate best model**

In [9]:
best_model = trainer.get_best_model()
X_test = test_clean.drop(columns=["price_usd"])
y_test = test_clean["price_usd"]

evaluator = Evaluator(best_model, X_test, y_test,
                      feature_names=X_test.columns.tolist())
evaluator.compute_metrics() \
         .plot_residuals() \
         .plot_feature_importance()

print("Metrics:", evaluator.get_metrics())

2026-06-13 14:29:21 | INFO     | evaluate | Evaluator initialized | test samples: 400
2026-06-13 14:29:21 | INFO     | evaluate | Evaluation Results:
2026-06-13 14:29:21 | INFO     | evaluate |   MAE   : 2742.67
2026-06-13 14:29:21 | INFO     | evaluate |   RMSE  : 3937.38
2026-06-13 14:29:21 | INFO     | evaluate |   R2    : 0.9864
2026-06-13 14:29:21 | INFO     | evaluate |   MAPE  : 3.7
2026-06-13 14:29:21 | INFO     | evaluate | ─────────────────
2026-06-13 14:29:21 | INFO     | evaluate | Residual plot saved → c:\Users\mrcoo\OneDrive\Desktop\ev-price-prediction\reports\residuals.png
2026-06-13 14:29:21 | INFO     | evaluate | Feature importance plot saved → c:\Users\mrcoo\OneDrive\Desktop\ev-price-prediction\reports\feature_importance.png
Metrics: {'MAE': 2742.67, 'RMSE': np.float64(3937.38), 'R2': 0.9864, 'MAPE': 3.7}
